In [1]:
import pandas as pd              # Loads and works with tabular data (DataFrames)
import numpy as np               # Performs numerical calculations and works with arrays
import matplotlib.pyplot as plt  # Creates charts and plots
import seaborn as sns            # Creates statistical charts with simpler styling
import joblib   # Saves and loads Python objects (like trained models) to and from disk
from pathlib import Path

from sklearn.model_selection import train_test_split  # Splits data into training and testing sets
from sklearn.pipeline import Pipeline  # Creates a pipeline of data transformations and model fitting
from sklearn.impute import SimpleImputer  # Fills in missing values in the data
from sklearn.compose import ColumnTransformer  # Applies different transformations to different columns
from sklearn.preprocessing import OneHotEncoder, StandardScaler  # Encodes categorical variables and scales numerical features

In [2]:
df = pd.read_csv("../data/processed/cleaned_data.csv")  # Loads data from a CSV file into a DataFrame
df.head()  # Displays the first few rows of the DataFrame

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [3]:
df.columns

Index(['MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street', 'Alley',
       'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope',
       'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
       'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle',
       'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea',
       'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond',
       'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating', 'HeatingQC',
       'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType', 'GarageYrBlt',
       'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond',
       'PavedDrive', 'Wo

### Total Bathroom (Merging)

In [4]:
df["TotalBathrooms"] = (
df["FullBath"]
+ 0.5 * df["HalfBath"]
+ df["BsmtFullBath"] 
+ 0.5 * df["BsmtHalfBath"]
)   # Creates a new column "TotalBathrooms" by combining full and half bathrooms from different parts of the house

df.sample() # Displays a random sample of rows from the DataFrame

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,TotalBathrooms
841,70,RM,60.0,10440,Pave,Grvl,Reg,Lvl,AllPub,Inside,...,NaN,NaN,NaN,0,6,2008,WD,Normal,157500,2.0


In [5]:
df["TotalSF"] = (
df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]
)

df.sample() # Displays a random sample of rows from the DataFrame

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,TotalBathrooms,TotalSF
678,20,RL,80.0,11844,Pave,NaN,IR1,Lvl,AllPub,Inside,...,NaN,NaN,0,7,2009,New,Partial,372500,2.5,4092


In [6]:
# df.loc[
#     df["Id"] == 120,
# ]   # Displays the row(s) in the DataFrame where the "Id" column has a value of 120

# df.loc[120]  # Displays the row in the DataFrame with index 120

In [7]:
df["TotalPorchSF"] = (
df["OpenPorchSF"] + df["EnclosedPorch"] + df["3SsnPorch"] + df["ScreenPorch"]
)

df.sample() # Displays a random sample of rows from the DataFrame

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,TotalBathrooms,TotalSF,TotalPorchSF
1074,20,RL,74.0,8556,Pave,NaN,Reg,Lvl,AllPub,Inside,...,NaN,0,5,2007,WD,Normal,194000,2.0,2480,93


In [8]:
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]  # Creates a new column "HouseAge" by calculating the difference between the year sold and the year built

In [9]:
df["RemodelAge"] = df["YrSold"] - df["YearRemodAdd"]  # Creates a new column "RemodelAge" by calculating the difference between the year sold and the year remodeled

### Separate Features (X) and Target (y)
_X contains the input features used by the model, while y contains the target variable we want to predict. Therefore, SalePrice is removed from X and stored separately in y._

In [10]:
X = df.drop("SalePrice", axis=1)  # Creates a new DataFrame X by dropping the "SalePrice" column from the original DataFrame df and axis 1 means we are dropping a column (axis=0 would mean dropping a row). This is typically done to separate the features (X) from the target variable (y) in a machine learning context.
y = df["SalePrice"]  # Creates a new Series y containing only the "SalePrice" column from the original DataFrame df.

In [11]:
X

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,MiscVal,MoSold,YrSold,SaleType,SaleCondition,TotalBathrooms,TotalSF,TotalPorchSF,HouseAge,RemodelAge
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,2,2008,WD,Normal,3.5,2566,61,5,5
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,...,0,5,2007,WD,Normal,2.5,2524,0,31,31
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,9,2008,WD,Normal,3.5,2706,42,7,6
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,2,2006,WD,Abnorml,2.0,2473,307,91,36
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,...,0,12,2008,WD,Normal,3.5,3343,84,8,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1455,60,RL,62.0,7917,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,8,2007,WD,Normal,2.5,2600,40,8,7
1456,20,RL,85.0,13175,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,2,2010,WD,Normal,3.0,3615,0,32,22
1457,70,RL,66.0,9042,Pave,NaN,Reg,Lvl,AllPub,Inside,...,2500,5,2010,WD,Normal,2.0,3492,60,69,4
1458,20,RL,68.0,9717,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,4,2010,WD,Normal,2.0,2156,112,60,14


In [12]:
y

0       208500
1       181500
2       223500
3       140000
4       250000
         ...  
1455    175000
1456    210000
1457    266500
1458    142125
1459    147500
Name: SalePrice, Length: 1460, dtype: int64

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [14]:
X_train
# X_test.head()   # Displays the first few rows of the X_test DataFrame, which contains the features for the test set.
# y_train
# y_test


,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,...,MiscVal,MoSold,YrSold,SaleType,SaleCondition,TotalBathrooms,TotalSF,TotalPorchSF,HouseAge,RemodelAge
254,20,RL,70.0,8400,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,6,2010,WD,Normal,2.0,2628,0,53,53
1066,60,RL,59.0,7837,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,5,2009,WD,Normal,2.5,2370,40,16,15
638,30,RL,67.0,8777,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,5,2008,WD,Normal,1.0,1592,164,98,58
799,50,RL,60.0,7200,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,6,2007,WD,Normal,2.5,2499,264,70,57
380,50,RL,50.0,5000,Pave,Pave,Reg,Lvl,AllPub,Inside,...,0,5,2010,WD,Normal,2.0,2717,242,86,60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1095,20,RL,78.0,9317,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,3,2007,WD,Normal,2.0,2628,22,1,1
1130,50,RL,65.0,7804,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,12,2009,WD,Normal,3.0,3103,44,81,59
1294,20,RL,60.0,8172,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,4,2006,WD,Normal,2.0,1728,0,51,16
860,50,RL,55.0,7642,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,6,2007,WD,Normal,1.5,2338,240,89,9


In [15]:
numerical_features = X_train.select_dtypes(include=[np.number]).columns.tolist()  # Selects the names of all numerical features in the training set
categorical_features = X_train.select_dtypes(include=[object]).columns.tolist()  # Selects the names of all categorical features in the training set

C:\Users\ACER\AppData\Local\Temp\ipykernel_41860\4208974176.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=[object]).columns.tolist()  # Selects the names of all categorical features in the training set


In [16]:
print(numerical_features)  # Prints the list of numerical feature names
print(categorical_features)  # Prints the list of categorical feature names

['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'TotalBathrooms', 'TotalSF', 'TotalPorchSF', 'HouseAge', 'RemodelAge']
['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual', 'Functional'

In [17]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # Fills in missing values in numerical features with the median value
    ("scaler", StandardScaler()),  # Scales numerical features to have a mean of 0 and a standard deviation of 1
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),  # Fills in missing values in categorical features with the most frequent value
    ("encoder", OneHotEncoder(handle_unknown="ignore")),  # Converts categorical features into a one-hot encoded format, ignoring any unknown categories during transformation
])

In [18]:
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),  # Applies the numerical pipeline to the numerical features
    ("cat", categorical_pipeline, categorical_features),  # Applies the categorical pipeline to the categorical features
])

In [19]:
model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

# Save the preprocessor before fitting
joblib.dump(preprocessor, model_dir / "preprocessor.pkl")   # Saves the preprocessor object to a file named "preprocessor.pkl" in the "../models" directory. This allows you to reuse the same preprocessing steps later without having to redefine them.

['..\\models\\preprocessor.pkl']